In [13]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START,END,MessagesState
from langchain_core.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv 


In [14]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")
MAX_WINDOW_SIZE=6

In [15]:
def get_windowed_message(messages: list) -> list:
    system_msgs = [msg for msg in messages if isinstance(msg, SystemMessage)]
    chat_msgs = [msg for msg in messages if not isinstance(msg, SystemMessage)]

    recent_chat_msgs = chat_msgs[-MAX_WINDOW_SIZE:]
    return system_msgs + recent_chat_msgs


In [16]:
def interviewer_node(state:MessagesState):
    windowed_list=get_windowed_message(state["messages"])
    print(f"\n[Window Tracker] Total in State: {len(state['messages'])} | Sent to LLM: {len(windowed_list)}")
    response=llm.invoke(windowed_list)
    return {"messages":[response]}

In [17]:
builder=StateGraph(MessagesState)

In [18]:
builder.add_node("interviewer_node",interviewer_node)
builder.add_edge(START,"interviewer_node")
builder.add_edge("interviewer_node",END)
persona_graph=builder.compile()

In [19]:
def run_persona_bot():
    print("Senior Python Interviewer Bot started! Type 'exit' or 'quit' to stop.")
    print()

    persona_prompt = SystemMessage(
        content=(
            "You are a strict Senior Python Technical Interviewer. "
            "Ask one short Python coding or concept question at a time. "
            "Evaluate the user's previous answer briefly in 1-2 sentences, "
            "and immediately ask the next question. Stay in character at all times."
        )
    )
    current_state = {"messages": [persona_prompt]}
    while True:
        user_input = input("Candidate (You): ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print("Interview ended. Good luck!")
            break

        if not user_input:
            continue

        current_state = persona_graph.invoke({
            "messages": current_state["messages"] + [HumanMessage(content=user_input)]
        })
        latest_ai_msg = current_state["messages"][-1]
        print()
        print(f"Interviewer: {latest_ai_msg.content}")
        print()
        print("-" * 50)


In [ ]:
if __name__ == "__main__":
    run_persona_bot()